# 🫀 Pipeline Pengenalan Pola — Heart Disease

**Alur:** Data → EDA → Cleaning → Normalisasi → Feature Selection → Bayesian Classifier → Evaluasi

| | Detail |
|---|---|
| Data latih | 20 sampel (extremely imbalanced) |
| Data uji | 5 sampel baru |
| Fitur | `age`, `cholesterol`, `max_hr`, `blood_pressure` |
| Label | `disease` — 0 = Sehat, 1 = Sakit |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 10})
print('✅ Library siap')

---
## 1 · Dataset

Data dibuat **acak/realistis** — bukan urut. Kelas sengaja **imbalanced ekstrem**: 17 Sehat vs 3 Sakit.  
Ada 2 nilai hilang (NaN) sebagai latihan cleaning.

In [ ]:
# ── DATA LATIH (20 sampel) ──────────────────────────────────
# Kelas 0 (Sehat) = 17 | Kelas 1 (Sakit) = 3  → rasio 85:15
# Nilai dibuat acak agar distribusi tidak terlalu rapi

train_data = {
    'age':           [63, 37, 55, 41, 70, 48, 33, 58, 45, 52,
                      29, 60, 44, 38, 67, 50, 35, 72, 68, 65],
    'cholesterol':   [188, 172, 221, 195, 207, 183, 165, 240, 199, 214,
                      158, 230, 201, 177, 218, 193, 169,  np.nan, 298, 281],
    'max_hr':        [163, 172, 148, 158, 121, 155, 174,  np.nan, 161, 145,
                      179, 132, 153, 170, 127, 157, 176, 108, 103, 112],
    'blood_pressure':[118, 112, 132, 121, 144, 124, 108, 138, 119, 129,
                      104, 141, 122, 115, 143, 126, 110, 162, 168, 155],
    'disease':       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
                        0,   0,   0,   0,   0,   0,   0,   1,   1,   1]
}

# ── DATA UJI (5 sampel baru, tidak ada di data latih) ──────
test_data = {
    'age':           [57, 40, 66, 31, 74],
    'cholesterol':   [225, 178, 310, 160, 275],
    'max_hr':        [138, 165, 107, 173, 109],
    'blood_pressure':[134, 114, 165, 109, 158],
    'disease':       [  0,   0,   1,   0,   1]   # label asli untuk evaluasi
}

df_train = pd.DataFrame(train_data)
df_test  = pd.DataFrame(test_data)

features = ['age', 'cholesterol', 'max_hr', 'blood_pressure']

print('── DATA LATIH ──────────────────────────────────────')
print(df_train.to_string())
print(f'\nShape: {df_train.shape}  |  Kelas: {dict(df_train.disease.value_counts().sort_index())}')
print('\n── DATA UJI ────────────────────────────────────────')
print(df_test.to_string())
print(f'\nShape: {df_test.shape}  |  Kelas: {dict(df_test.disease.value_counts().sort_index())}')

---
## 2 · EDA — Jelajahi Data

In [ ]:
# ── 2a. Distribusi Kelas ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
fig.suptitle('Distribusi Kelas — Extremely Imbalanced', fontweight='bold')

cnt = df_train['disease'].value_counts().sort_index()
colors = ['#4CAF50', '#F44336']
lbl    = ['Sehat (0)', 'Sakit (1)']

# Bar
bars = axes[0].bar(lbl, cnt.values, color=colors, edgecolor='black', width=0.45)
for b, v in zip(bars, cnt.values):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.3, str(v), ha='center', fontweight='bold')
axes[0].set_ylim(0, 22)
axes[0].set_title('Jumlah per Kelas')

# Pie
axes[1].pie(cnt.values, labels=lbl, colors=colors, autopct='%1.0f%%',
            startangle=90, explode=(0, 0.1), wedgeprops={'edgecolor':'black'})
axes[1].set_title(f'Proporsi (Rasio {cnt[0]/cnt[1]:.0f}:1)')

plt.tight_layout(); plt.show()
print(f'⚠️  Imbalance ratio = {cnt[0]/cnt[1]:.1f}:1  — kelas Sakit hanya {cnt[1]/cnt.sum():.0%}')

In [ ]:
# ── 2b. Distribusi Fitur per Kelas ───────────────────────────
feat_labels = ['Usia (thn)', 'Kolesterol (mg/dL)', 'Detak Maks (bpm)', 'Tek. Darah (mmHg)']

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle('Distribusi Tiap Fitur — Sehat vs Sakit', fontweight='bold')

for ax, feat, label in zip(axes.flat, features, feat_labels):
    for cls, clr in [(0,'#4CAF50'), (1,'#F44336')]:
        vals = df_train[df_train.disease==cls][feat].dropna()
        ax.hist(vals, bins=7, alpha=0.6, color=clr, edgecolor='black',
                label=f'{"Sehat" if cls==0 else "Sakit"} (μ={vals.mean():.0f})')
        ax.axvline(vals.mean(), color=clr, lw=2, ls='--')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print('💡 Fitur dengan mean yang paling berbeda antar kelas = paling diskriminatif')

In [ ]:
# ── 2c. Statistik Deskriptif & Missing Values ────────────────
print('── Statistik Deskriptif (Data Latih) ──')
print(df_train.describe().round(1).to_string())

print('\n── Missing Values ──')
mv = df_train.isnull().sum()
print(mv[mv > 0].to_string())
print(f'Total: {mv.sum()} sel kosong dari {df_train.size}')

---
## 3 · Data Cleaning

**Strategi:** Imputasi dengan **median per kelas** — bukan median global agar tidak mencampur distribusi kelas.

In [ ]:
# ── Imputasi Median per Kelas ────────────────────────────────
df_clean = df_train.copy()

for feat in features:
    for idx in df_clean[df_clean[feat].isnull()].index:
        cls = df_clean.loc[idx, 'disease']
        med = df_clean[df_clean.disease == cls][feat].median()
        df_clean.loc[idx, feat] = med
        print(f'  Imputasi baris {idx:>2} | {feat:<16} kelas={cls} → {med:.1f}')

print(f'\n✅ Missing tersisa: {df_clean.isnull().sum().sum()}')

# Visualisasi sebelum vs sesudah (scatter age vs cholesterol)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
fig.suptitle('Sebelum vs Sesudah Imputasi (age vs cholesterol)', fontweight='bold')
for ax, (data, title) in zip(axes, [(df_train,'Sebelum (ada NaN)'), (df_clean,'Sesudah (bersih)')]):
    for cls, clr, mrk in [(0,'#4CAF50','o'), (1,'#F44336','^')]:
        s = data[data.disease==cls]
        ax.scatter(s.age, s.cholesterol, c=clr, marker=mrk, s=70,
                   edgecolors='black', lw=0.7, label=f'{"Sehat" if cls==0 else "Sakit"}')
    ax.set_xlabel('Age'); ax.set_ylabel('Cholesterol')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## 4 · Normalisasi Min-Max

$$x_{norm} = \dfrac{x - x_{min}}{x_{max} - x_{min}} \quad \Rightarrow \quad \text{hasil} \in [0, 1]$$

**Penting:** parameter min/max dihitung dari **data latih**, lalu diterapkan ke data uji.

In [ ]:
X_tr = df_clean[features].values.astype(float)
y_tr = df_clean['disease'].values
X_te = df_test[features].values.astype(float)
y_te = df_test['disease'].values

# Hitung dari training set saja
x_min = X_tr.min(axis=0)
x_max = X_tr.max(axis=0)

X_tr_n = (X_tr - x_min) / (x_max - x_min)
X_te_n = (X_te - x_min) / (x_max - x_min)   # pakai param TRAIN

print('Parameter normalisasi (dari data latih):')
print(f'{"Fitur":<18} {"min":>8} {"max":>8}')
for i, f in enumerate(features):
    print(f'  {f:<16} {x_min[i]:>8.1f} {x_max[i]:>8.1f}')

# Boxplot perbandingan
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribusi Fitur: Sebelum vs Sesudah Normalisasi', fontweight='bold')
for ax, data, title in zip(axes, [X_tr, X_tr_n], ['Sebelum', 'Sesudah [0,1]']):
    bp = ax.boxplot(data, labels=features, patch_artist=True,
                    medianprops={'color':'red','lw':2})
    for patch in bp['boxes']:
        patch.set_facecolor('#B3E5FC')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

---
## 5 · Feature Selection

Pilih **top-3 fitur** terbaik menggunakan:
- **Fisher Score** — seberapa jauh mean antar kelas?
- **Korelasi |Pearson|** — seberapa linear hubungannya dengan label?

In [ ]:
fisher, corr = {}, {}

print(f'{"Fitur":<18} {"μ Sehat":>8} {"μ Sakit":>8} {"Fisher":>9} {"Korelasi":>10}')
print('-'*57)
for i, feat in enumerate(features):
    x0 = X_tr_n[y_tr==0, i];  x1 = X_tr_n[y_tr==1, i]
    fs = (x1.mean()-x0.mean())**2 / (x0.var()+x1.var()+1e-9)
    cr = abs(np.corrcoef(X_tr_n[:,i], y_tr)[0,1])
    fisher[feat]=fs; corr[feat]=cr
    print(f'  {feat:<16} {x0.mean():>8.3f} {x1.mean():>8.3f} {fs:>9.4f} {cr:>10.4f}')

# Rata-rata ranking
rank_f = {k:r+1 for r,(k,_) in enumerate(sorted(fisher.items(), key=lambda x:x[1], reverse=True))}
rank_c = {k:r+1 for r,(k,_) in enumerate(sorted(corr.items(),   key=lambda x:x[1], reverse=True))}
avg_rk = sorted({f:(rank_f[f]+rank_c[f])/2 for f in features}.items(), key=lambda x:x[1])

top3 = [f for f,_ in avg_rk[:3]]
print(f'\n✅ Top-3 Fitur Terpilih: {top3}')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Skor Feature Selection', fontweight='bold')
for ax, scores, title in zip(axes,
    [fisher, corr], ['Fisher Score (↑ lebih diskriminatif)', '|Korelasi Pearson| dengan Label']):
    srt = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    nm, vl = zip(*srt)
    clrs = ['#F44336' if n in top3 else '#90A4AE' for n in nm]
    bars = ax.barh(nm, vl, color=clrs, edgecolor='black')
    ax.invert_yaxis()
    for b, v in zip(bars, vl):
        ax.text(v+0.001, b.get_y()+b.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
    ax.set_title(title, fontweight='bold')
    p = mpatches.Patch(color='#F44336', label='Terpilih'); ax.legend(handles=[p], fontsize=8)
plt.tight_layout(); plt.show()

fi = [features.index(f) for f in top3]
X_tr_sel = X_tr_n[:, fi]
X_te_sel = X_te_n[:, fi]
print(f'Shape train: {X_tr_sel.shape} | Shape test: {X_te_sel.shape}')

---
## 6 · Bayesian Decision Classifier (Manual)

$$\hat{y} = \arg\max_k \; P(C_k) \cdot \prod_i P(x_i \mid C_k)$$

Setiap fitur diasumsikan distribusi **Gaussian** → hitung μ dan σ² per kelas dari data latih.

In [ ]:
class BayesGaussian:
    """Gaussian Naive Bayes — implementasi manual."""

    def fit(self, X, y):
        self.classes = np.unique(y)
        n = len(y)
        self.prior, self.mu, self.var = {}, {}, {}
        for c in self.classes:
            Xc = X[y==c]
            self.prior[c] = len(Xc) / n
            self.mu[c]    = Xc.mean(axis=0)
            self.var[c]   = Xc.var(axis=0) + 1e-9   # smoothing
        return self

    def _log_likelihood(self, X, c):
        """Log P(x | C) = sum log Gaussian(x_i; μ_i, σ²_i)"""
        mu, var = self.mu[c], self.var[c]
        return -0.5 * np.sum(np.log(2*np.pi*var) + (X - mu)**2/var, axis=1)

    def predict_proba(self, X):
        log_post = np.column_stack([
            np.log(self.prior[c]) + self._log_likelihood(X, c)
            for c in self.classes
        ])
        # Softmax untuk konversi ke probabilitas
        e = np.exp(log_post - log_post.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes[np.argmax(self.predict_proba(X), axis=1)]


model = BayesGaussian().fit(X_tr_sel, y_tr)

print('── Parameter Model ──────────────────────────────────')
for c in model.classes:
    print(f'\n  Kelas {c} ({"Sehat" if c==0 else "Sakit"}) | Prior={model.prior[c]:.4f}')
    for i, f in enumerate(top3):
        print(f'    {f:<18}: μ={model.mu[c][i]:.4f}  σ²={model.var[c][i]:.6f}')
print('\n✅ Model selesai dilatih!')

In [ ]:
# Visualisasi kurva Gaussian per fitur
x_plot = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, len(top3), figsize=(13, 3.5))
fig.suptitle('Distribusi Gaussian per Kelas (Parameter dari Training)', fontweight='bold')

for ax, (feat, idx) in zip(axes, enumerate(top3)):
    for c, clr, lbl in [(0,'#4CAF50','Sehat'), (1,'#F44336','Sakit')]:
        mu, var = model.mu[c][idx], model.var[c][idx]
        pdf = (1/np.sqrt(2*np.pi*var)) * np.exp(-0.5*(x_plot-mu)**2/var)
        ax.plot(x_plot, pdf, color=clr, lw=2.5, label=f'{lbl} μ={mu:.2f}')
        ax.fill_between(x_plot, pdf, alpha=0.1, color=clr)
        ax.axvline(mu, color=clr, ls='--', lw=1.2)
        # Titik data asli di bawah
        pts = X_tr_sel[y_tr==c, idx]
        ax.scatter(pts, np.zeros_like(pts)-0.2, c=clr, marker='|', s=150, lw=1.5)
    ax.set_title(feat, fontweight='bold'); ax.set_xlabel('[0,1]'); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 7 · Prediksi & Evaluasi

Evaluasi dilakukan pada **data uji (5 sampel)** — data yang belum pernah dilihat model.

In [ ]:
# ── Prediksi ─────────────────────────────────────────────────
y_pred  = model.predict(X_te_sel)
y_proba = model.predict_proba(X_te_sel)

print('── Hasil Prediksi Data Uji (5 sampel) ──────────────────')
print(f'{"No":>3} {"Label Asli":>12} {"Prediksi":>10} {"P(Sehat)":>10} {"P(Sakit)":>10} {"":>8}')
print('-'*57)
for i in range(len(y_te)):
    asli = 'Sehat' if y_te[i]==0 else 'Sakit'
    pred = 'Sehat' if y_pred[i]==0 else 'Sakit'
    ok   = '✅' if y_te[i]==y_pred[i] else '❌'
    print(f'{i+1:>3} {asli:>12} {pred:>10} {y_proba[i,0]:>10.4f} {y_proba[i,1]:>10.4f} {ok:>8}')

In [ ]:
# ── Confusion Matrix Manual ───────────────────────────────────
TP = int(((y_te==1)&(y_pred==1)).sum())
TN = int(((y_te==0)&(y_pred==0)).sum())
FP = int(((y_te==0)&(y_pred==1)).sum())
FN = int(((y_te==1)&(y_pred==0)).sum())

precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
recall    = TP/(TP+FN) if (TP+FN)>0 else 0.0
f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
accuracy  = (TP+TN)/len(y_te)

print('── Confusion Matrix ──────────────────────────────────')
print('                     Prediksi')
print('                  Sehat(0)  Sakit(1)')
print(f'  Asli  Sehat(0)  TN={TN}       FP={FP}')
print(f'        Sakit(1)  FN={FN}       TP={TP}')
print()
print(f'  Accuracy  = {accuracy:.2f}   ({TP+TN}/{len(y_te)} benar)')
print(f'  Precision = {precision:.2f}   TP/(TP+FP) = {TP}/({TP}+{FP})')
print(f'  Recall    = {recall:.2f}   TP/(TP+FN) = {TP}/({TP}+{FN})')
print(f'  F1-Score  = {f1:.2f}   harmonic mean P & R')
print(f'\n  ⚠️  FN = {FN} — pasien sakit yang TIDAK terdeteksi (berbahaya!)')

In [ ]:
# ── Visualisasi Evaluasi ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Evaluasi Model pada Data Uji (5 sampel)', fontweight='bold')

# 1. Confusion Matrix
ax = axes[0]
cm = np.array([[TN, FP],[FN, TP]])
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=3)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred Sehat','Pred Sakit'])
ax.set_yticklabels(['Asli Sehat','Asli Sakit'])
lbl_cm = [['TN','FP'],['FN','TP']]
for i in range(2):
    for j in range(2):
        clr = 'white' if cm[i,j]>1 else 'black'
        ax.text(j, i, f'{lbl_cm[i][j]}\n{cm[i,j]}', ha='center', va='center',
                fontsize=13, fontweight='bold', color=clr)
ax.set_title('Confusion Matrix', fontweight='bold')
# Kotak merah di FN
ax.add_patch(plt.Rectangle((-0.5,0.5),1,1,lw=2.5,ec='red',fc='none'))

# 2. Metrik Bar
ax2 = axes[1]
names  = ['Accuracy','Precision','Recall','F1-Score']
values = [accuracy, precision, recall, f1]
clrs2  = ['#2196F3','#FF9800','#E91E63','#9C27B0']
bars   = ax2.bar(names, values, color=clrs2, edgecolor='black', width=0.5)
ax2.set_ylim(0, 1.2)
for b, v in zip(bars, values):
    ax2.text(b.get_x()+b.get_width()/2, v+0.03, f'{v:.2f}',
             ha='center', fontweight='bold', fontsize=11)
ax2.axhline(1.0, color='gray', ls='--', alpha=0.4)
ax2.set_title('Metrik Evaluasi', fontweight='bold')

# 3. Probabilitas P(Sakit) tiap data uji
ax3 = axes[2]
x_idx = np.arange(len(y_te))
p_sakit = y_proba[:,1]
bclrs = ['#F44336' if p>=0.5 else '#4CAF50' for p in p_sakit]
ax3.bar(x_idx, p_sakit, color=bclrs, edgecolor='black', width=0.5)
ax3.axhline(0.5, color='black', ls='--', lw=2, label='Threshold 0.5')
ax3.set_xticks(x_idx)
ax3.set_xticklabels([f'T{i+1}\n({"S" if y_te[i]==1 else "H"})' for i in range(len(y_te))])
ax3.set_ylabel('P(Sakit | x)'); ax3.set_ylim(0, 1.1)
ax3.set_title('Probabilitas Prediksi\n(H/S = label asli)', fontweight='bold')
for i, (p, yp) in enumerate(zip(p_sakit, y_pred)):
    ok = '✓' if y_te[i]==yp else '✗'
    clr = 'green' if ok=='✓' else 'red'
    ax3.text(i, p+0.04, ok, ha='center', fontsize=13, color=clr, fontweight='bold')
ax3.legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 8 · Decision Boundary (2 Fitur Terbaik)

In [ ]:
# ── Decision Boundary (latih ulang dengan 2 fitur terbaik) ───
f1n, f2n = top3[0], top3[1]
f1i, f2i = 0, 1
X2_tr = X_tr_sel[:, [f1i, f2i]]
X2_te = X_te_sel[:, [f1i, f2i]]

import io, sys
buf = io.StringIO(); sys.stdout = buf
m2 = BayesGaussian().fit(X2_tr, y_tr)
sys.stdout = sys.__stdout__

h  = 0.008
xx, yy = np.meshgrid(np.arange(-0.05, 1.1, h), np.arange(-0.05, 1.1, h))
Z  = m2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlGn')
ax.contour(xx, yy, Z, levels=[0.5], colors='black', lw=2, linestyles='--')

# Plot data train
for c, clr, mrk, lbl in [(0,'#2E7D32','o','Train Sehat'), (1,'#B71C1C','^','Train Sakit')]:
    m = y_tr==c
    ax.scatter(X2_tr[m,0], X2_tr[m,1], c=clr, marker=mrk, s=80,
               edgecolors='black', lw=0.7, label=lbl, zorder=4)

# Plot data test — lebih besar & tebal
for c, clr, mrk, lbl in [(0,'#4CAF50','o','Test Sehat'), (1,'#F44336','^','Test Sakit')]:
    m = y_te==c
    ax.scatter(X2_te[m,0], X2_te[m,1], c=clr, marker=mrk, s=180,
               edgecolors='black', lw=2, label=lbl, zorder=5)

# Tandai prediksi salah pada data test
wrong = y_te != m2.predict(X2_te)
if wrong.any():
    ax.scatter(X2_te[wrong,0], X2_te[wrong,1], s=350, facecolors='none',
               edgecolors='blue', lw=2.5, label='Test Salah Prediksi', zorder=6)

ax.set_xlabel(f'{f1n} [norm]', fontsize=11)
ax.set_ylabel(f'{f2n} [norm]', fontsize=11)
ax.set_title('Decision Boundary — Bayesian Classifier\n(data train kecil, data test besar)',
             fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout(); plt.show()
print('💡 Batas keputusan (garis putus) memisahkan zona Sehat (hijau) dan Sakit (merah)')

---
## 9 · Ringkasan & Insight

| Langkah | Teknik |
|---|---|
| Cleaning | Imputasi median per kelas |
| Normalisasi | Min-Max (parameter dari train) |
| Feature Selection | Fisher Score + Korelasi Pearson |
| Classifier | Gaussian Naive Bayes (manual) |
| Evaluasi | Precision · Recall · F1 · Accuracy |

### 💡 Insight Utama

1. **Accuracy bisa menipu** pada data imbalanced — model naif yang selalu prediksi "Sehat" dapat Accuracy 85%!
2. **Recall kelas Sakit** adalah metrik terpenting di kasus medis — FN (pasien sakit tidak terdeteksi) jauh lebih berbahaya dari FP.
3. **Prior Bayesian** otomatis memperhitungkan ketidakseimbangan kelas melalui $P(C_k)$.
4. **Feature Selection** membuang noise dan mempercepat komputasi tanpa kehilangan informasi penting.

---
### 🧪 Coba Sendiri
- Ubah threshold dari `0.5` → `0.3` → apa yang terjadi pada Recall?
- Ganti 1 data uji yang salah prediksi — apa yang membuatnya salah?
- Coba hanya dengan 1 fitur terbaik — seberapa jauh penurunan F1?